# R3-M1 experiment audit
Run this notebook on a Colab T4 after syncing `experiments/r3_m1/` and the project to Drive. This audit reads train and selection audio only; it does not train a model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, sys, subprocess, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Update if necessary.
SHARDS = ROOT / 'data/processed/cohort_v1'  # Or the canonical shard copy on scratch.
EXPERIMENT = ROOT / 'experiments/r3_m1'
assert (EXPERIMENT / 'scripts/audit.py').is_file() and SHARDS.is_dir()
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
tests = [str(EXPERIMENT / 'tests/test_audit.py'), str(EXPERIMENT / 'tests/test_checkpoint_guard.py'), str(ROOT / 'tests/test_r3_bigru_frontend.py')]
run = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], capture_output=True, text=True)
print(run.stdout, run.stderr)
assert run.returncode == 0 and ' skipped' not in run.stdout, 'Resolve fixture failures before proceeding'


In [ ]:
report = EXPERIMENT / 'reports' / ('audit_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
cmd = [sys.executable, str(EXPERIMENT / 'scripts/audit.py'), '--root', str(ROOT), '--shard-dir', str(SHARDS), '--output', str(report)]
result = subprocess.run(cmd, text=True)
print('Audit exit code:', result.returncode, '(0=passed, 1=failed, 2=incomplete)')
print('Share this report:', report)
if report.exists():
    print('Overall:', json.loads(report.read_text(encoding='utf-8'))['overall'])
